<a href="https://colab.research.google.com/github/DhariniVenki/CS769-Bias-Content-Moderation/blob/main/tagging/Tag_Posts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tagging swaps and control synonyms

The proposal's tagging step. An 8B LLM (Qwen3-8B) tags each held-out post in `verification_300.csv` with:

- **swaps**: every co-referring word and modifier that marks the group, with its counterpart
  (she/her/woman → he/his/man, "deaf cousin" → "cousin");
- **synonyms**: neutral replacements for other words in the post, from which the framework builds the
  matched-magnitude controls.

The tags feed Δ directly, so each one is checked twice:

1. **Automatic checks** (here): every phrase occurs in the post; after the swap, no word of the original
   group is left; the post does not also mention the other gender; synonyms never touch swap, identity or
   toxic words; there are enough synonyms to match the swap's size.
2. **Review** (people + Gemini) on `tagging_review.csv`, then `finalize` writes the verified evaluation set.

Runtime: GPU. On an L4/A100 the model runs in bf16. On a T4 it loads in 4-bit, and 300 posts take
roughly 15–25 minutes.

## 1. Setup

In [ ]:
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
DATA_REPO = "https://github.com/DhariniVenki/CS769-Bias-Content-Moderation.git"
DATA_REPO_DIR = "/content/CS769-Bias-Content-Moderation"
FRAMEWORK_REPO = "https://github.com/shreyasnat2804/bias-compare-poc.git"
FRAMEWORK_BRANCH = "claude/testing-framework-llm-bert-njfxy2"
FRAMEWORK_DIR = "/content/bias-compare"

def clone_or_pull(url, path, branch=None):
    if os.path.isdir(path):
        subprocess.run(["git", "-C", path, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q"] + (["-b", branch] if branch else []) + [url, path], check=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    clone_or_pull(DATA_REPO, DATA_REPO_DIR)
    clone_or_pull(FRAMEWORK_REPO, FRAMEWORK_DIR, FRAMEWORK_BRANCH)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", FRAMEWORK_DIR, "bitsandbytes"], check=True)
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN secret: fine for Qwen3-8B (public).")
else:
    DATA_REPO_DIR = os.path.abspath(".")

sys.path.insert(0, f"{DATA_REPO_DIR}/tagging")
import tag_posts

## 2. Configuration

`LIMIT = 10` tags 10 posts first, to look at the output before the full run. Set it to `None` for all 300.

In [ ]:
DATA_DIR = "/content/drive/MyDrive/CS769/data/processed" if IN_COLAB else "data/processed"
INPUT = f"{DATA_DIR}/verification_300.csv"
OUT_DIR = "/content/drive/MyDrive/kqv/tagging" if IN_COLAB else "results/tagging"

MODEL = "Qwen/Qwen3-8B"
QUANTIZATION = "auto"   # "auto": bf16 with >= 22 GB of GPU memory, else 4-bit
BATCH_SIZE = 8
LIMIT = 10

## 3. Tag

In [ ]:
import argparse
args = argparse.Namespace(input=INPUT, out_dir=OUT_DIR, model=MODEL, quantization=QUANTIZATION,
                          batch_size=BATCH_SIZE, limit=LIMIT)
tagged, review = tag_posts.tag(args)

## 4. Inspect

`auto_ok` rows passed every automatic check. Read the `swapped_text` column: it is the counterfactual the
models will see. Typical problems: a missed pronoun (`incomplete_swap`), a second referent of the other
gender (`mentions_*_too`), and a disability word not about a person (`llm_unusable`).

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 200)
rv = pd.DataFrame(review)
display(rv.groupby("axis")["auto_ok"].agg(["sum", "count"]).rename(columns={"sum": "auto_ok", "count": "rows"}))
display(rv[["id", "axis", "auto_ok", "issues", "text", "swapped_text", "swaps", "synonyms"]].head(20))

The framework's own preview of the auto-passing (still **unverified**) rows: the exact swap and controls
it would score.

In [ ]:
from biascompare import preview_perturbations

autopass = f"{OUT_DIR}/tagged_posts_autopass.jsonl"
for axis in ("gender", "disability"):
    rows = preview_perturbations(autopass, axis=axis, controls="tagged", n_controls=5, seed=0)
    print(f"{axis}: {len(rows)} rows, {sum(bool(r['flags']) for r in rows)} with flags")
    display(pd.DataFrame([{"id": r["id"], "group": r["group"], "original": r["original"], "swap": r["swap"]["text"],
                           "words changed": r["n_words_changed"], "controls": len(r["controls"]),
                           "first control": r["controls"][0]["text"] if r["controls"] else None,
                           "flags": r["flags"]} for r in rows]).head(15))

## 5. Review (people + Gemini)

Open `tagging_review.csv` (in `OUT_DIR`) in Google Sheets. The `annotators` column says who reviews each
row: `human` rows go to us, `gemini` rows to Gemini, `both` rows to both (100 posts).

**Two people on the `both` rows.** Inter-human agreement is the ceiling for human-model agreement, so on
the 100 `both` rows one teammate fills `human_verdict` and another fills `human2_verdict`, each without
looking at the other's column. `finalize` reports human vs human, human vs Gemini and second human vs
Gemini (Cohen's κ). Rows where the two people disagree are left out until they agree, after which they
write the settled verdict in both columns.

For each row write a verdict in `human_verdict` (and `human2_verdict`) or `gemini_verdict`:

- `accept`: the swap is complete and correct, and the synonyms are neutral;
- `fix`: put corrected JSON in `corrected_swaps` and/or `corrected_synonyms` (same format as the
  `swaps` / `synonyms` columns);
- `reject`: the post is unusable on this axis (several referents, not about a person, unfixable).

When people and Gemini both reviewed a row, the human verdict is the one used. Download the sheet as CSV into `OUT_DIR` as
`tagging_review_filled.csv`, then run the next cell.

In [ ]:
fargs = argparse.Namespace(review=f"{OUT_DIR}/tagging_review_filled.csv", tagged=f"{OUT_DIR}/tagged_posts.jsonl",
                           out=f"{OUT_DIR}/verified_posts.jsonl")
verified, agreement = tag_posts.finalize(fargs)

`verified_posts.jsonl` is the evaluation set: set `DATASET` to it in `bias_study.ipynb`. That notebook's
preview cell and `docs/DATA.md` §6 list the checks to run before a real run.